dataset link: https://huggingface.co/datasets/QCRI/CrisisMMD -> need to log in first
- multimodal data baseline paper: https://arxiv.org/abs/2004.11838
another dataset link: [quanml0703/CrisisMMD_v2](https://huggingface.co/datasets/quanml0703/CrisisMMD_v2) with [associated paper](https://arxiv.org/pdf/2004.11838)
- merge the train, dev (validation), test datasets
- remove the cases where labels disagree

In [17]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
from huggingface_hub import hf_hub_download
import pandas as pd, csv

In [ ]:
dataset = load_dataset("QCRI/CrisisMMD", "informative")
label_names = dataset["train"].features["label"].names

full = concatenate_datasets([dataset[s] for s in ["train", "dev", "test"]])
print(len(full), "rows total")
print(pd.Series(full["event_name"]).value_counts())

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


18082 rows total
hurricane_maria         4556
hurricane_irma          4504
hurricane_harvey        4434
california_wildfires    1589
mexico_earthquake       1380
srilanka_floods         1022
iraq_iran_earthquake     597
Name: count, dtype: int64


In [ ]:
events = ["california_wildfires", "hurricane_harvey", "hurricane_irma", "hurricane_maria",
          "iraq_iran_earthquake", "mexico_earthquake", "srilanka_floods"]

ann = pd.concat([
    pd.read_csv(hf_hub_download("QCRI/CrisisMMD", f"files_individual_events/{e}_final_data.tsv",
                                repo_type="dataset"),
                sep="\t", quoting=csv.QUOTE_NONE, dtype=str)
    for e in events
])
print(ann.columns.tolist())
print(ann[["text_info", "image_info"]].value_counts())

agreed_ids = set(ann.loc[ann["text_info"] == ann["image_info"], "image_id"])
print(len(agreed_ids), "agreed image_ids")

california_wildfires_final_data.tsv:   0%|          | 0.00/619k [00:00<?, ?B/s]

hurricane_harvey_final_data.tsv:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

hurricane_irma_final_data.tsv:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

hurricane_maria_final_data.tsv:   0%|          | 0.00/1.73M [00:00<?, ?B/s]

iraq_iran_earthquake_final_data.tsv:   0%|          | 0.00/229k [00:00<?, ?B/s]

mexico_earthquake_final_data.tsv:   0%|          | 0.00/531k [00:00<?, ?B/s]

srilanka_floods_final_data.tsv:   0%|          | 0.00/369k [00:00<?, ?B/s]

['tweet_id', 'image_id', 'text_info', 'text_info_conf', 'image_info', 'image_info_conf', 'text_human', 'text_human_conf', 'image_human', 'image_human_conf', 'image_damage', 'image_damage_conf', 'tweet_text', 'image_url', 'image_path']
text_info        image_info     
informative      informative        8445
                 not_informative    4410
not_informative  not_informative    4298
                 informative         929
Name: count, dtype: int64
12743 agreed image_ids


In [22]:
agreed = full.filter(lambda iid: iid in agreed_ids, input_columns="image_id")
print(len(agreed), "agreed rows")

lab = dict(zip(ann["image_id"], ann["image_info"]))
mismatch = sum(label_names[l] != lab[i] for i, l in zip(agreed["image_id"], agreed["label"]))
print("label mismatches:", mismatch)

12743 agreed rows
label mismatches: 22


In [23]:
new = DatasetDict({
    "train": agreed.filter(lambda e: "earthquake" not in e and e != "california_wildfires",
                           input_columns="event_name"),
    "dev":   agreed.filter(lambda e: "earthquake" in e, input_columns="event_name"),
    "test":  agreed.filter(lambda e: e == "california_wildfires", input_columns="event_name"),
})
print(new)

Filter:   0%|          | 0/12743 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12743 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12743 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image_path', 'image', 'label'],
        num_rows: 9917
    })
    dev: Dataset({
        features: ['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image_path', 'image', 'label'],
        num_rows: 1621
    })
    test: Dataset({
        features: ['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image_path', 'image', 'label'],
        num_rows: 1205
    })
})


# downloading the model

In [ ]:
import os
from huggingface_hub import snapshot_download

os.environ["HF_HOME"] = "./models"   # set before importing transformers

for m in ["google/vit-base-patch16-224-in21k",
          "google/vit-large-patch16-224-in21k",
          "google/vit-huge-patch14-224-in21k"]:
    snapshot_download(m)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

{"timestamp":"2026-11-17T23:37:34.698223Z","level":"ERROR","fields":{"message":"Error logging to file \"/work/users/r/a/raina/hf_cache/xet/logs/xet_20261117T183734696-0500_24760.log\" (Read-only file system (os error 30)); falling back to console logging."},"filename":"/Users/runner/work/xet-core/xet-core/xet_runtime/src/logging/init.rs","line_number":64}


OSError: I/O error: I/O error: Read-only file system (os error 30)